# 02 — Statistical Analysis

This notebook runs the formal statistical tests that notebook 01's EDA
only hinted at, and uses their results to make one concrete modelling
decision: **do we forecast price levels or returns?**

Each test below is included because it directly informs a modelling
choice later in the project — not run just because it exists. For every
test we state: why we're using it, its null hypothesis, the result, and
what that result means for the modelling decisions ahead.

In [1]:
import sys
import warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")  # statsmodels is noisy about upcoming API changes; not relevant here

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from scipy.stats import jarque_bera

from config import SECTOR_STOCKS, ALL_TICKERS, TICKER_TO_SECTOR, PROCESSED_DIR, TABLES_DIR, FIGURES_DIR

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

prices = pd.read_csv(PROCESSED_DIR / "prices_processed.csv", parse_dates=["Date"])

## 1. Stationarity: ADF and KPSS

**Why.** Almost every classical time-series model (ARIMA, and the whole
idea of "features computed over a stable process" for the ML/DL models)
assumes or works much better with a stationary series. If we forecast a
non-stationary series directly, a model can score well simply by
tracking the trend, without learning anything genuinely predictive.

We run two complementary tests, because they check the same thing from
opposite directions and a well-known quirk is that they can disagree:

* **ADF (Augmented Dickey-Fuller).** H0: the series has a unit root
  (is non-stationary). A low p-value lets us reject H0, i.e. conclude
  stationarity.
* **KPSS.** H0: the series *is* (trend-)stationary. A low p-value lets us
  reject H0, i.e. conclude non-stationarity — the opposite direction
  from ADF.

Running both avoids relying on a single test's assumptions.

In [2]:
def adf_kpss(series: pd.Series) -> dict:
    series = series.dropna()
    adf_stat, adf_p, *_ = adfuller(series, autolag="AIC")
    try:
        kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
    except Exception:
        kpss_stat, kpss_p = np.nan, np.nan
    return {"ADF stat": adf_stat, "ADF p-value": adf_p,
            "KPSS stat": kpss_stat, "KPSS p-value": kpss_p}


stationarity_rows = []
for ticker in ALL_TICKERS:
    d = prices[prices["Ticker"] == ticker]
    price_result = adf_kpss(d["Close"])
    return_result = adf_kpss(d["LogReturn"])
    stationarity_rows.append({
        "Ticker": ticker, "Sector": TICKER_TO_SECTOR[ticker], "Series": "Price",
        **price_result,
    })
    stationarity_rows.append({
        "Ticker": ticker, "Sector": TICKER_TO_SECTOR[ticker], "Series": "Log return",
        **return_result,
    })

stationarity = pd.DataFrame(stationarity_rows)
stationarity["ADF: stationary (p<0.05)"] = stationarity["ADF p-value"] < 0.05
stationarity["KPSS: stationary (p>0.05)"] = stationarity["KPSS p-value"] > 0.05
stationarity.to_csv(TABLES_DIR / "stationarity_tests.csv", index=False)
stationarity.round(4)

/tmp/ipykernel_2351/2512140331.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is smaller than the p-value returned.
  kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
/tmp/ipykernel_2351/2512140331.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is greater than the p-value returned.
  kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
/tmp/ipykernel_2351/2512140331.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is smaller than the p-value returned.
  kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")


/tmp/ipykernel_2351/2512140331.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is greater than the p-value returned.
  kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
/tmp/ipykernel_2351/2512140331.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is smaller than the p-value returned.
  kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
/tmp/ipykernel_2351/2512140331.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is greater than the p-value returned.
  kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")


/tmp/ipykernel_2351/2512140331.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is smaller than the p-value returned.
  kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
/tmp/ipykernel_2351/2512140331.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is greater than the p-value returned.
  kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
/tmp/ipykernel_2351/2512140331.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is smaller than the p-value returned.
  kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")


/tmp/ipykernel_2351/2512140331.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is greater than the p-value returned.
  kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
/tmp/ipykernel_2351/2512140331.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is smaller than the p-value returned.
  kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
/tmp/ipykernel_2351/2512140331.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is greater than the p-value returned.
  kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")


/tmp/ipykernel_2351/2512140331.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is smaller than the p-value returned.
  kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
/tmp/ipykernel_2351/2512140331.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is smaller than the p-value returned.
  kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")


/tmp/ipykernel_2351/2512140331.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is greater than the p-value returned.
  kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
/tmp/ipykernel_2351/2512140331.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is smaller than the p-value returned.
  kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
/tmp/ipykernel_2351/2512140331.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is greater than the p-value returned.
  kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")


/tmp/ipykernel_2351/2512140331.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is smaller than the p-value returned.
  kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
/tmp/ipykernel_2351/2512140331.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is greater than the p-value returned.
  kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
/tmp/ipykernel_2351/2512140331.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is smaller than the p-value returned.
  kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
/tmp/ipykernel_2351/2512140331.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is greater than the

/tmp/ipykernel_2351/2512140331.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is smaller than the p-value returned.
  kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
/tmp/ipykernel_2351/2512140331.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is greater than the p-value returned.
  kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")


,Ticker,Sector,Series,ADF stat,ADF p-value,KPSS stat,KPSS p-value,ADF: stationary (p<0.05),KPSS: stationary (p>0.05)
0,HDFCBANK,Banking,Price,-1.9623,0.3033,7.5853,0.0100,False,False
1,HDFCBANK,Banking,Log return,-11.4179,0.0000,0.2584,0.1000,True,True
2,ICICIBANK,Banking,Price,0.1966,0.9721,8.2561,0.0100,False,False
3,ICICIBANK,Banking,Log return,-12.6816,0.0000,0.0276,0.1000,True,True
4,SBIN,Banking,Price,0.4544,0.9834,7.4137,0.0100,False,False
5,SBIN,Banking,Log return,-13.5711,0.0000,0.0625,0.1000,True,True
6,TCS,IT,Price,-1.5557,0.5058,6.5767,0.0100,False,False
7,TCS,IT,Log return,-52.2479,0.0000,0.3380,0.1000,True,True
8,INFY,IT,Price,-1.4085,0.5782,6.8349,0.0100,False,False
9,INFY,IT,Log return,-19.5758,0.0000,0.2088,0.1000,True,True


In [3]:
summary = (
    stationarity.groupby("Series")[["ADF: stationary (p<0.05)", "KPSS: stationary (p>0.05)"]]
    .mean()
    .rename(columns=lambda c: c + " (share of 12 stocks)")
)
summary

,ADF: stationary (p<0.05) (share of 12 stocks),KPSS: stationary (p>0.05) (share of 12 stocks)
Series,,
Log return,1.0,1.0
Price,0.0,0.0


**Result.** For price levels, ADF fails to reject the unit-root null for
essentially every stock, and KPSS rejects stationarity for essentially
every stock — both tests agree prices are non-stationary. For log
returns, ADF strongly rejects the unit-root null and KPSS fails to
reject stationarity for essentially every stock — both tests agree
returns are (at least approximately) stationary.

**What this means for modelling.** Forecasting raw price levels would
mean forecasting a non-stationary series, where a naive "no change"
prediction is already hard to beat by construction and any model can
look artificially good just by following the trend. Log returns are
statistically much better behaved and are the series ARIMA and the
stationarity assumptions behind the ML/DL feature engineering actually
expect. **We forecast log returns as the primary target for every model
in this project**, and convert return forecasts back to a price
trajectory only where it's useful for interpretation (e.g. the
actual-vs-predicted price charts and the simple backtest in notebook
06). We don't duplicate the whole project with a parallel price-forecasting
track — the stationarity evidence here is clear enough that it isn't
needed, and ARIMA's own differencing step in notebook 03 effectively
confirms the same conclusion once more.

## 2. Autocorrelation of returns: Ljung-Box

**Why.** Notebook 01's ACF plot suggested returns are close to white
noise. The Ljung-Box test makes this precise: it jointly tests whether
the first *k* autocorrelations are all zero. This tells us how much
linear structure, if any, a model like ARIMA has to work with in the raw
returns.

H0: the return series shows no autocorrelation up to lag *k* (white
noise).

In [4]:
ljung_rows = []
for ticker in ALL_TICKERS:
    d = prices[prices["Ticker"] == ticker]
    lb = acorr_ljungbox(d["LogReturn"], lags=[10, 20], return_df=True)
    ljung_rows.append({
        "Ticker": ticker, "Sector": TICKER_TO_SECTOR[ticker],
        "LB stat (lag 10)": lb.loc[10, "lb_stat"], "p-value (lag 10)": lb.loc[10, "lb_pvalue"],
        "LB stat (lag 20)": lb.loc[20, "lb_stat"], "p-value (lag 20)": lb.loc[20, "lb_pvalue"],
    })
ljung = pd.DataFrame(ljung_rows)
ljung.to_csv(TABLES_DIR / "ljung_box_returns.csv", index=False)
ljung.round(4)

,Ticker,Sector,LB stat (lag 10),p-value (lag 10),LB stat (lag 20),p-value (lag 20)
0,HDFCBANK,Banking,33.3886,0.0002,54.0594,0.0001
1,ICICIBANK,Banking,26.6438,0.0030,65.8216,0.0000
2,SBIN,Banking,27.7021,0.0020,61.0026,0.0000
3,TCS,IT,10.5435,0.3942,35.3427,0.0184
4,INFY,IT,25.7226,0.0041,42.2281,0.0026
5,WIPRO,IT,4.9338,0.8956,19.8245,0.4690
6,SUNPHARMA,Healthcare,9.9361,0.4461,14.1478,0.8229
7,DRREDDY,Healthcare,12.0909,0.2790,29.7391,0.0742
8,CIPLA,Healthcare,19.4736,0.0346,28.2020,0.1047
9,MARUTI,Automotive,10.0518,0.4360,24.2531,0.2315


**Result.** Roughly half the stocks show statistically significant
autocorrelation at the 5% level at one or both lags, but the effect
sizes (autocorrelation coefficients, not shown in the summary table but
visible in notebook 01's ACF plot) are small. This is a realistic,
mild-efficiency-violation result for large, liquid stocks — not the
strong, exploitable structure the p-values alone might suggest. **This
means ARIMA has, at best, a small amount of genuine linear structure to
exploit, and we should not expect it to dramatically beat a random walk**
— a useful expectation to carry into notebook 03's results.

## 3. Distribution: Jarque-Bera

**Why.** Several of our evaluation choices (e.g. using RMSE, which is
sensitive to outliers) and modelling choices implicitly assume something
about the error/return distribution. Jarque-Bera checks whether returns
are normally distributed, based on their sample skewness and kurtosis.

H0: the series is normally distributed.

In [5]:
jb_rows = []
for ticker in ALL_TICKERS:
    d = prices[prices["Ticker"] == ticker]
    stat, p = jarque_bera(d["LogReturn"].dropna())
    jb_rows.append({
        "Ticker": ticker, "Sector": TICKER_TO_SECTOR[ticker],
        "Skewness": d["LogReturn"].skew(), "Excess kurtosis": d["LogReturn"].kurtosis(),
        "JB stat": stat, "JB p-value": p,
    })
jb = pd.DataFrame(jb_rows)
jb.to_csv(TABLES_DIR / "jarque_bera_returns.csv", index=False)
jb.round(4)

,Ticker,Sector,Skewness,Excess kurtosis,JB stat,JB p-value
0,HDFCBANK,Banking,-0.3292,10.3256,11712.6571,0.0
1,ICICIBANK,Banking,-0.0963,10.4368,11921.7771,0.0
2,SBIN,Banking,0.4535,13.2239,19225.0717,0.0
3,TCS,IT,-0.1042,3.8842,1654.1941,0.0
4,INFY,IT,-0.6208,9.2934,9618.1255,0.0
5,WIPRO,IT,0.2412,6.8879,5215.2209,0.0
6,SUNPHARMA,Healthcare,-0.1408,5.1815,2944.9170,0.0
7,DRREDDY,Healthcare,0.2083,6.6526,4860.0186,0.0
8,CIPLA,Healthcare,0.6836,5.3745,3364.1798,0.0
9,MARUTI,Automotive,-0.1161,10.5262,12128.9098,0.0


**Result.** Jarque-Bera rejects normality (p < 0.001) for all 12 stocks,
and every stock shows positive excess kurtosis (fat tails — large moves
are far more common than a normal distribution would predict), matching
the well-known "fat tails" stylised fact of daily equity returns and the
COVID-era extreme days flagged in notebook 01.

**What this means for modelling.** We should not assume Gaussian errors
anywhere, and should be cautious about metrics that are very sensitive to
a handful of extreme days. This is exactly why the evaluation in this
project uses **MAE and directional accuracy as primary metrics, not just
RMSE** — MAE is far less distorted by the fat tails than RMSE, and
directional accuracy asks a question (up or down?) that doesn't depend
on the return distribution's shape at all.

## 4. Volatility clustering: ARCH test

**Why.** Notebook 01's rolling-volatility plot showed visible clustering
(calm periods, turbulent periods). Engle's ARCH-LM test makes this
formal: it regresses squared returns on their own lags and tests whether
that regression has any explanatory power. If it does, volatility today
depends on volatility recently — the defining feature that GARCH-family
models are built to capture.

H0: no ARCH effect (squared returns are not autocorrelated, i.e.
volatility is not clustering).

In [6]:
arch_rows = []
for ticker in ALL_TICKERS:
    d = prices[prices["Ticker"] == ticker]
    stat, p, _, _ = het_arch(d["LogReturn"].dropna(), nlags=10)
    arch_rows.append({"Ticker": ticker, "Sector": TICKER_TO_SECTOR[ticker],
                       "ARCH-LM stat": stat, "p-value": p})
arch_test = pd.DataFrame(arch_rows)
arch_test.to_csv(TABLES_DIR / "arch_test_returns.csv", index=False)
arch_test.round(4)

,Ticker,Sector,ARCH-LM stat,p-value
0,HDFCBANK,Banking,606.5089,0.0
1,ICICIBANK,Banking,382.3642,0.0
2,SBIN,Banking,85.6613,0.0
3,TCS,IT,233.4586,0.0
4,INFY,IT,148.4611,0.0
5,WIPRO,IT,65.1344,0.0
6,SUNPHARMA,Healthcare,186.9431,0.0
7,DRREDDY,Healthcare,55.1694,0.0
8,CIPLA,Healthcare,156.3559,0.0
9,MARUTI,Automotive,586.2338,0.0


**Result.** The ARCH-LM test rejects the no-clustering null (p < 0.001)
for all 12 stocks — strong, unambiguous evidence of volatility
clustering, consistent with the EDA.

**What this means for modelling.** Volatility clustering is a real,
well-established feature of this data. But — and this is the point made
in the project brief — **that does not automatically mean GARCH belongs
in the main model comparison.** GARCH models the *variance* of returns,
not their conditional *mean*; it is not built to predict the sign or
magnitude of tomorrow's return the way ARIMA/RF/XGBoost/LSTM/CNN/
Transformer are. Forcing GARCH into the same price/return-forecasting
comparison as the other seven models would be comparing it on a task it
isn't designed for. Given that the ARCH test here confirms clustering is
real and worth investigating, we include a **small, separate GARCH
volatility-forecasting side-experiment in notebook 06**, evaluated on
its own terms (forecasting volatility, not price/return direction),
rather than bolting it onto the main leaderboard.

## 5. Sector-level summary

In [7]:
fig, ax = plt.subplots(figsize=(9, 4))
sector_order = list(SECTOR_STOCKS.keys())
sns.boxplot(data=jb.merge(arch_test, on=["Ticker", "Sector"], suffixes=("_jb", "_arch")),
            x="Sector", y="Excess kurtosis", order=sector_order, ax=ax)
ax.set_title("Excess kurtosis of daily log returns, by sector")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "07_kurtosis_by_sector.png", bbox_inches="tight")
plt.show()

## 6. Summary of decisions made in this notebook

| Test | Result | Decision it drives |
|---|---|---|
| ADF + KPSS | Prices non-stationary, returns stationary | **Forecast log returns, not price levels**, for every model |
| Ljung-Box | Weak-to-moderate autocorrelation in returns | Don't expect ARIMA to dramatically beat the random walk |
| Jarque-Bera | Returns non-normal, fat-tailed | Use MAE and directional accuracy as primary metrics, not RMSE alone |
| ARCH-LM | Strong volatility clustering, all 12 stocks | Justifies a small, separate GARCH volatility side-experiment — not a place in the main model leaderboard |

These decisions carry forward into every remaining notebook.